# AI-powered Movie Recommendation System

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

print(ratings.head())
print(movies.head())

print(ratings.info())
print(movies.info())

## Data Cleaning and Preprocessing

In [ ]:
# Remove unnecessary columns
ratings_clean = ratings.drop(columns=['timestamp'])
movies_clean = movies.drop(columns=['genres'])

In [ ]:
print("Missing values in Ratings dataset:\n", ratings_clean.isnull().sum())
print("Missing values in Movies dataset:\n", movies_clean.isnull().sum())

In [ ]:
print(ratings_clean.head())
print(movies_clean.head())

print(ratings_clean.info())
print(movies_clean.info())

## How many movies has each user rated?

In [ ]:
user_ratings_count = ratings_clean.groupby("userId").size()

print(f"Average number of rated movies per user: {user_ratings_count.mean():.2f}")
print("Distribution of rated movies per user:")
print(user_ratings_count.describe())

plt.figure(figsize=(10,5), dpi=150)
plt.hist(user_ratings_count, bins=100, edgecolor='black')
plt.xlabel("Number of Rated Movies")
plt.ylabel("Number of Users")
plt.title("Distribution of Rated Movies per User")
plt.yscale('log') 
plt.show()

## How many times has each movie been rated?

In [ ]:
movie_ratings_count = ratings_clean.groupby("movieId").size()

print(f"Average number of ratings per movie: {movie_ratings_count.mean():.2f}")
print("Distribution of ratings per movie:")
print(movie_ratings_count.describe())

plt.figure(figsize=(10,5), dpi=150)
plt.hist(movie_ratings_count, bins=100, edgecolor='black')
plt.xlabel("Number of Ratings per Movie")
plt.ylabel("Number of Movies")
plt.title("Distribution of Ratings per Movie")
plt.yscale('log')
plt.show()

### Data Analysis Results and Interpretation

**Average Number of Rated Movies per User: 153.81**

- On average, users have rated 154 movies.
- Each user has rated at least 20 movies (min: 20).
- Some users have rated up to 32,202 movies, indicating highly active users..

**Average Number of Ratings per Movie: 423.39**

- A movie is rated an average of 423 times.
- However, some movies have received only 1-2 ratings (min: 1).
- The most-rated movie has 81,491 ratings! (most likely a very popular movie).
- 50% (median) value is just 6 → meaning half of the movies have been rated 6 times or less.

**What does this mean?**
- Movies with very few ratings make reliable recommendations difficult.
- We should filter out movies with extremely low ratings (e.g., keep only movies with at least 20-30 ratings).

# Data Cleaning – Filtering Movies with Few Ratings

In [ ]:
# Check rating distribution percentiles
percentiles = np.percentile(movie_ratings_count, [10, 25, 50, 75, 90, 95, 99])
print(f"Rating distribution percentiles:\n{percentiles}")

**Rating Percentiles per Movie**

- 10th percentile: 1 rating → 10% of movies have been rated only once.
- 25th percentile: 2 ratings → 25% of movies have received 2 or fewer ratings.
- 50th percentile (Median): 6 ratings → Half of the movies have received 6 or fewer ratings.
- 75th percentile: 36 ratings → 75% of movies have received at least 36 ratings.
- 90th percentile: 413 ratings → Top 10% of movies have received at least 413 ratings.
- 95th percentile: 1,503 ratings → Top 5% of movies have received at least 1,503 ratings.
- 99th percentile: 9,941 ratings → Top 1% of movies have received 9,941 or more ratings.

**Decision:** To improve recommendation reliability, we filter out movies with fewer than 10 ratings. Since the median rating count is 6, keeping only movies with at least 10 ratings ensures we retain about 40-45% of movies in the system.

In [ ]:
min_votes = 10
popular_movies = movie_ratings_count[movie_ratings_count >= min_votes].index

# Filter ratings to include only popular movies
filtered_ratings = ratings_clean[ratings_clean["movieId"].isin(popular_movies)]

print(f"📌 Filtered Ratings Dataset size: {filtered_ratings.shape}")
print(f"📌 Number of unique movies after filtering: {filtered_ratings['movieId'].nunique()}")
print(f"📌 Number of unique users after filtering: {filtered_ratings['userId'].nunique()}")

## Training & Evaluating the SVD Model

In [ ]:
# !pip install scikit-surprise

In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

reader = Reader(rating_scale=(0.5, 5.0)) 
data = Dataset.load_from_df(filtered_ratings[["userId", "movieId", "rating"]], reader)

trainset, testset = train_test_split(data, test_size=0.2)

model = SVD()
model.fit(trainset)

predictions = model.test(testset)
rmse = accuracy.rmse(predictions)

print(f"📌 RMSE Value: {rmse:.4f}")

## Computing Similarity Between Movies (Cosine Similarity)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

small_ratings = filtered_ratings[filtered_ratings["userId"] <= 5000]  # First 5000 users
user_movie_matrix = small_ratings.pivot(index="userId", columns="movieId", values="rating").fillna(0)

movie_similarity = cosine_similarity(user_movie_matrix.T)

movie_similarity_df = pd.DataFrame(movie_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)

movie_id = 1
similar_movies = movie_similarity_df[movie_id].sort_values(ascending=False).iloc[1:11]  # Top 10 similar movies

recommended_movies = movies[movies["movieId"].isin(similar_movies.index)]
print(f"📌 Most similar movies to {movies[movies.movieId == movie_id]['title'].values[0]}:")
print(recommended_movies)

**What Do These Results Mean?**

Movies with high similarity scores are typically:
- Same genre (Animation, Family, Adventure, etc.)
- Released in the same era
- Watched by similar audiences


With this approach, we can add a **"Similar Movies"** feature to our recommendation system!

# Personalized Movie Recommendations for Users

In [ ]:
# Select a User ID for Recommendation
user_id = 1

user_movies = filtered_ratings[filtered_ratings["userId"] == user_id]["movieId"].tolist()

all_movies = filtered_ratings["movieId"].unique()
unrated_movies = [movie for movie in all_movies if movie not in user_movies]

predictions = [model.predict(user_id, movie) for movie in unrated_movies]
predictions.sort(key=lambda x: x.est, reverse=True)

top_recommendations = predictions[:10]
top_movie_ids = [pred.iid for pred in top_recommendations]
recommended_movies = movies[movies["movieId"].isin(top_movie_ids)]

print(f"📌 Recommended movies for User {user_id}:")
print(recommended_movies)

### Interpretation of Recommendations
- The recommended movies often consist of classic, high-rated, and award-winning films.
- The list is mostly dominated by Drama, but some Documentary and Comedy movies are also included.
- There are also diverse genre recommendations based on the user's preferences.

## Personalizing Recommendations Based on User Preferences

How Our Recommendation System Works:
- Find the user's most preferred genres based on past ratings.
- Select only movies with at least 50 ratings to improve speed and reliability.
- Filter these movies further to the top 1000 for efficiency and recommendation quality.
- Use the trained SVD model to predict ratings and recommend the top 10 movies.

In [ ]:
from collections import Counter

user_id = 1 

user_movies = filtered_ratings[filtered_ratings["userId"] == user_id]["movieId"].tolist()
user_genres = movies[movies["movieId"].isin(user_movies)]["genres"].str.split("|").sum()

top_genres = Counter(user_genres).most_common(3)
favorite_genres = [genre[0] for genre in top_genres]

print(f"📌 Most preferred genres for User {user_id}: {favorite_genres}")

popular_movies = filtered_ratings["movieId"].value_counts()
unrated_movies = popular_movies[popular_movies > 50].index.tolist()
unrated_movies = [movie for movie in unrated_movies if movie not in user_movies]
unrated_movies = unrated_movies[:1000]

predictions = [model.predict(user_id, movie) for movie in unrated_movies]
predictions.sort(key=lambda x: x.est, reverse=True)

filtered_recommendations = []
for pred in predictions:
    movie_id = pred.iid
    movie_genres = movies[movies["movieId"] == movie_id]["genres"].values
    if any(genre in movie_genres[0] for genre in favorite_genres):
        filtered_recommendations.append(pred)

top_recommendations = filtered_recommendations[:10]
top_movie_ids = [pred.iid for pred in top_recommendations]
recommended_movies = movies[movies["movieId"].isin(top_movie_ids)]

print(f"📌 Recommended movies for User {user_id} based on {favorite_genres}:")
print(recommended_movies)

## Testing Recommendations for Different Users

In [ ]:
# Select a different User ID for testing
user_id = 50

user_movies = filtered_ratings[filtered_ratings["userId"] == user_id]["movieId"].tolist()
user_genres = movies[movies["movieId"].isin(user_movies)]["genres"].str.split("|").sum()

from collections import Counter
top_genres = Counter(user_genres).most_common(3)
favorite_genres = [genre[0] for genre in top_genres]
print(f"📌 Most preferred genres for User {user_id}: {favorite_genres}")

popular_movies = filtered_ratings["movieId"].value_counts()
unrated_movies = popular_movies[popular_movies > 50].index.tolist()
unrated_movies = [movie for movie in unrated_movies if movie not in user_movies]
unrated_movies = unrated_movies[:1000]

predictions = [model.predict(user_id, movie) for movie in unrated_movies]
predictions.sort(key=lambda x: x.est, reverse=True)

filtered_recommendations = []
for pred in predictions:
    movie_id = pred.iid
    movie_genres = movies[movies["movieId"] == movie_id]["genres"].values
    if any(genre in movie_genres[0] for genre in favorite_genres):
        filtered_recommendations.append(pred)

top_recommendations = filtered_recommendations[:10]
top_movie_ids = [pred.iid for pred in top_recommendations]
recommended_movies = movies[movies["movieId"].isin(top_movie_ids)]

print(f"📌 Recommended movies for User {user_id} based on {favorite_genres}:")
print(recommended_movies)

### Results Analysis
- The recommended movies for User 50 are mainly Drama, Comedy, and Action, aligning with their preferences.
- The system maintains a balance between Comedy and Action-Drama recommendations.
- Django Unchained (Action|Drama|Western) fits well with the user’s Action preference.
- Life Is Beautiful and Intouchables are categorized as Comedy|Drama, making them logical choices.
- Shawshank Redemption and Green Mile, both Drama movies, validate the recommendation accuracy.

### The Recommendation System Works!
- The system accurately analyzes different users' genre preferences.
- The recommended movies are logical and relevant.
- The model successfully personalizes recommendations based on past viewing history.